In [ ]:
import os
from pathlib import Path

import pandas as pd
from pymongo import MongoClient

from logger import mylogger

cwd = Path(os.getcwd())
mylogger.debug(cwd)
root = cwd.parent
mylogger.debug(root)

folder = root / Path("data")
mylogger.info(folder)

dict_df = {}

for file in folder.glob("*.csv"):
    mylogger.info("\n" + "=" * 60)
    mylogger.info(f"FILE: {file.name}")
    mylogger.info("=" * 60)

    df = pd.read_csv(file)
    name_short = os.path.splitext(file.name)[0]
    dict_df[name_short] = df

    mylogger.info(f"Shape: {df.shape}")
    mylogger.info("\nColumns:")
    mylogger.info(df.columns.tolist())

    mylogger.info("\nData types:")
    mylogger.info(df.dtypes)

    mylogger.info("\nMissing values:")
    mylogger.info(df.isna().sum())

    mylogger.info("\nFirst 5 rows:")
    mylogger.info(df.head())

    mylogger.info("\nNumeric statistics:")
    mylogger.info(df.describe())

    duplicates = df.duplicated().sum()

    mylogger.info("\n--- Duplicates ---")
    mylogger.info(f"Duplicate rows: {duplicates:,}")

    for col in df.columns:
        duplicate_keys = df[col].duplicated().sum()
        mylogger.info(f"{col}: {duplicate_keys:,} duplicate values")


c:\Users\Utilisateur\Desktop\liste_depots\projet_groupe_mongodb\src
c:\Users\Utilisateur\Desktop\liste_depots\projet_groupe_mongodb
c:\Users\Utilisateur\Desktop\liste_depots\projet_groupe_mongodb\data

FILE: olist_customers_dataset.csv
Shape: (99441, 5)

Columns:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Data types:
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

Missing values:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

First 5 rows:
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd0871

In [3]:
# add 'orders' to mongoDb
client = MongoClient("localhost:27017")
mylogger.debug(dict_df.keys())
db = client["projet_groupe"]
db["orders"].drop()
orders_col = db["orders"]

orders_col.insert_many(dict_df["olist_orders_dataset"].to_dict("records"))
res = orders_col.find({}).limit(5)
for doc in res:
    mylogger.debug(doc)
orders_col.create_index("order_id")


dict_keys(['olist_customers_dataset', 'olist_geolocation_dataset', 'olist_orders_dataset', 'olist_order_items_dataset', 'olist_order_payments_dataset', 'olist_order_reviews_dataset', 'olist_products_dataset', 'olist_sellers_dataset', 'product_category_name_translation'])
{'_id': ObjectId('6aabbf2bb85471bfb9bda1d0'), 'order_id': 'e481f51cbdc54678b7cc49136f2d6af7', 'customer_id': '9ef432eb6251297304e76186b10a928d', 'order_status': 'delivered', 'order_purchase_timestamp': '2017-10-02 10:56:33', 'order_approved_at': '2017-10-02 11:07:15', 'order_delivered_carrier_date': '2017-10-04 19:55:00', 'order_delivered_customer_date': '2017-10-10 21:25:13', 'order_estimated_delivery_date': '2017-10-18 00:00:00'}
{'_id': ObjectId('6aabbf2bb85471bfb9bda1d1'), 'order_id': '53cdb2fc8bc7dce0b6741e2150273451', 'customer_id': 'b0830fb4747a6c6d20dea0b8c802d7ef', 'order_status': 'delivered', 'order_purchase_timestamp': '2018-07-24 20:41:37', 'order_approved_at': '2018-07-26 03:24:27', 'order_delivered_carrie

'order_id_1'

In [ ]:
# aggregate 'order_payments' into 'orders'
db["payments_temp"].drop()
payments_col = db["payments_temp"]
payments_col.insert_many(dict_df["olist_order_payments_dataset"].to_dict("records"))
payments_col.create_index("order_id")

pipeline = [
    {
        "$lookup": {
            "from": "payments_temp",
            "localField": "order_id",
            "foreignField": "order_id",
            "pipeline": [{"$project": {"_id": 0}}],
            "as": "payments",
        }
    },
    {"$unwind": "$payments"},
    {"$set": {"_id": "$$REMOVE"}},
    {"$out": "orders"},
]


orders_col.aggregate(pipeline)
res = orders_col.find({}).limit(5)
for doc in res:
    mylogger.debug(doc)
payments_col.drop()


{'_id': ObjectId('6aabbf3c342e44e218e028e5'), 'order_id': 'e481f51cbdc54678b7cc49136f2d6af7', 'customer_id': '9ef432eb6251297304e76186b10a928d', 'order_status': 'delivered', 'order_purchase_timestamp': '2017-10-02 10:56:33', 'order_approved_at': '2017-10-02 11:07:15', 'order_delivered_carrier_date': '2017-10-04 19:55:00', 'order_delivered_customer_date': '2017-10-10 21:25:13', 'order_estimated_delivery_date': '2017-10-18 00:00:00', 'payments': {'_id': ObjectId('6aabbf38b85471bfb9bf5053'), 'order_id': 'e481f51cbdc54678b7cc49136f2d6af7', 'payment_sequential': 1, 'payment_type': 'credit_card', 'payment_installments': 1, 'payment_value': 18.12}}
{'_id': ObjectId('6aabbf3c342e44e218e028e6'), 'order_id': 'e481f51cbdc54678b7cc49136f2d6af7', 'customer_id': '9ef432eb6251297304e76186b10a928d', 'order_status': 'delivered', 'order_purchase_timestamp': '2017-10-02 10:56:33', 'order_approved_at': '2017-10-02 11:07:15', 'order_delivered_carrier_date': '2017-10-04 19:55:00', 'order_delivered_customer_

In [ ]:
# aggregate 'order_items' into 'orders'

db["items_temp"].drop()
items_col = db["items_temp"]
items_col.insert_many(dict_df["olist_order_items_dataset"].to_dict("records"))
items_col.create_index("order_id")

pipeline = [
    {
        "$lookup": {
            "from": "items_temp",
            "localField": "order_id",
            "foreignField": "order_id",
            "pipeline": [{"$project": {"_id": 0}}],
            "as": "items",
        }
    },
    {"$unwind": "$items"},
    {"$set": {"_id": "$$REMOVE"}},
    {"$out": "orders"},
]


orders_col.aggregate(pipeline)
res = orders_col.find({}).limit(5)
for doc in res:
    mylogger.debug(doc)
items_col.drop()


{'_id': ObjectId('6aabbf49342e44e218e1beb3'), 'order_id': 'e481f51cbdc54678b7cc49136f2d6af7', 'customer_id': '9ef432eb6251297304e76186b10a928d', 'order_status': 'delivered', 'order_purchase_timestamp': '2017-10-02 10:56:33', 'order_approved_at': '2017-10-02 11:07:15', 'order_delivered_carrier_date': '2017-10-04 19:55:00', 'order_delivered_customer_date': '2017-10-10 21:25:13', 'order_estimated_delivery_date': '2017-10-18 00:00:00', 'payments': {'_id': ObjectId('6aabbf38b85471bfb9bf5053'), 'order_id': 'e481f51cbdc54678b7cc49136f2d6af7', 'payment_sequential': 1, 'payment_type': 'credit_card', 'payment_installments': 1, 'payment_value': 18.12}, 'items': {'_id': ObjectId('6aabbf46b85471bfb9c245c0'), 'order_id': 'e481f51cbdc54678b7cc49136f2d6af7', 'order_item_id': 1, 'product_id': '87285b34884572647811a353c7ac498a', 'seller_id': '3504c0cb71d7fa48d967e0e4c94d59d9', 'shipping_limit_date': '2017-10-06 11:07:15', 'price': 29.99, 'freight_value': 8.72}}
{'_id': ObjectId('6aabbf49342e44e218e1beb4

In [ ]:
# aggregate 'order_reviews' into 'orders'

db["reviews_temp"].drop()
reviews_col = db["reviews_temp"]
reviews_col.insert_many(dict_df["olist_order_reviews_dataset"].to_dict("records"))
reviews_col.create_index("order_id")

pipeline = [
    {
        "$lookup": {
            "from": "reviews_temp",
            "localField": "order_id",
            "foreignField": "order_id",
            "pipeline": [{"$project": {"_id": 0}}],
            "as": "reviews",
        }
    },
    {"$unwind": "$reviews"},
    {"$set": {"_id": "$$REMOVE"}},
    {"$out": "orders"},
]


orders_col.aggregate(pipeline)
res = orders_col.find({}).limit(5)
for doc in res:
    mylogger.debug(doc)

reviews_col.drop()

{'_id': ObjectId('6aabbf57342e44e218e38a14'), 'order_id': 'e481f51cbdc54678b7cc49136f2d6af7', 'customer_id': '9ef432eb6251297304e76186b10a928d', 'order_status': 'delivered', 'order_purchase_timestamp': '2017-10-02 10:56:33', 'order_approved_at': '2017-10-02 11:07:15', 'order_delivered_carrier_date': '2017-10-04 19:55:00', 'order_delivered_customer_date': '2017-10-10 21:25:13', 'order_estimated_delivery_date': '2017-10-18 00:00:00', 'payments': {'_id': ObjectId('6aabbf38b85471bfb9bf5053'), 'order_id': 'e481f51cbdc54678b7cc49136f2d6af7', 'payment_sequential': 1, 'payment_type': 'credit_card', 'payment_installments': 1, 'payment_value': 18.12}, 'items': {'_id': ObjectId('6aabbf46b85471bfb9c245c0'), 'order_id': 'e481f51cbdc54678b7cc49136f2d6af7', 'order_item_id': 1, 'product_id': '87285b34884572647811a353c7ac498a', 'seller_id': '3504c0cb71d7fa48d967e0e4c94d59d9', 'shipping_limit_date': '2017-10-06 11:07:15', 'price': 29.99, 'freight_value': 8.72}, 'reviews': {'_id': ObjectId('6aabbf54b8547

In [8]:
# aggregate 'customer'(shold be named order_to_customer) into 'orders'

db["customers_temp"].drop()
customers_col = db["customers_temp"]
dict_df["olist_customers_dataset"].rename(
    {"customer_id": "order_to_customer_id", "customer_unique_id": "customer_id"}
)
customers_col.insert_many(dict_df["olist_customers_dataset"].to_dict("records"))
customers_col.create_index("customer_id")

pipeline = [
    {
        "$lookup": {
            "from": "customers_temp",
            "localField": "customer_id",
            "foreignField": "customer_id",
            "pipeline": [{"$project": {"_id": 0}}],
            "as": "order_to_customer",
        }
    },
    {"$unwind": "$order_to_customer"},
    {"$set": {"_id": "$$REMOVE"}},
    {"$out": "orders"},
]


orders_col.aggregate(pipeline)
res = orders_col.find({}).limit(5)
for doc in res:
    mylogger.debug(doc)

customers_col.drop()

{'_id': ObjectId('6aabde4a342e44e218e55465'), 'order_id': 'e481f51cbdc54678b7cc49136f2d6af7', 'customer_id': '9ef432eb6251297304e76186b10a928d', 'order_status': 'delivered', 'order_purchase_timestamp': '2017-10-02 10:56:33', 'order_approved_at': '2017-10-02 11:07:15', 'order_delivered_carrier_date': '2017-10-04 19:55:00', 'order_delivered_customer_date': '2017-10-10 21:25:13', 'order_estimated_delivery_date': '2017-10-18 00:00:00', 'payments': {'_id': ObjectId('6aabbf38b85471bfb9bf5053'), 'order_id': 'e481f51cbdc54678b7cc49136f2d6af7', 'payment_sequential': 1, 'payment_type': 'credit_card', 'payment_installments': 1, 'payment_value': 18.12}, 'items': {'_id': ObjectId('6aabbf46b85471bfb9c245c0'), 'order_id': 'e481f51cbdc54678b7cc49136f2d6af7', 'order_item_id': 1, 'product_id': '87285b34884572647811a353c7ac498a', 'seller_id': '3504c0cb71d7fa48d967e0e4c94d59d9', 'shipping_limit_date': '2017-10-06 11:07:15', 'price': 29.99, 'freight_value': 8.72}, 'reviews': {'_id': ObjectId('6aabbf54b8547